In [ ]:
#!/usr/bin/env python3
# encoding: utf-8
#
# Copyright (C) 2024 Max Planck Institute for Multidisclplinary Sciences
# Copyright (C) 2024 Bharti Arora <bharti.arora@mpinat.mpg.de>
# Copyright (C) 2024 Ajinkya Kulkarni <ajinkya.kulkarni@mpinat.mpg.de>
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Affero General Public License as
# published by the Free Software Foundation, either version 3 of the
# License, or (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Affero General Public License for more details.
#
# You should have received a copy of the GNU Affero General Public License
# along with this program. If not, see <https://www.gnu.org/licenses/>.

########################################################################################

In [ ]:
import sys
# Don't generate the __pycache__ folder locally
sys.dont_write_bytecode = True
# Print exception without the built-in python warning
sys.tracebacklimit = 0

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from modules_maldi import *

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
#
# imzML file exported from Bruker FlexImaging 5.1:
#   File → Export → imzML  (both .imzML and .ibd must be in the same folder)
#
imzml_path       = 'Data/sample.imzML'

# QuPath annotation export: File → Export → Annotations as GeoJSON
# Annotated classes used in the paper: LSCC, RSCC,
#   high_collagen_coherence, low_collagen_coherence,
#   high_nuclei_distribution, low_nuclei_distribution
geojson_path     = 'Data/qupath_annotations.geojson'

# Region names that match the annotation class names in QuPath
region_a_name    = 'LSCC'
region_b_name    = 'RSCC'

# Peak detection
min_intensity_fraction = 0.01   # medium noise reduction (SCiLS default)
ppm_tolerance          = 100.0  # integration window in ppm

# ROC thresholds (from the paper)
auc_high  = 0.6
auc_low   = 0.4

# PCA
n_pca_components = 5

# Bisecting k-means
n_clusters = 6

# Output folder
output_folder = 'Results_MALDI'
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# ── Step 1: Load imzML and apply TIC normalisation ─────────────────────────────
mz_array, intensity_matrix, coordinates, image_shape = read_imzml(imzml_path)

print(f'Spectra loaded   : {intensity_matrix.shape[0]}')
print(f'M/z bins         : {intensity_matrix.shape[1]}')
print(f'M/z range        : {mz_array.min():.1f} – {mz_array.max():.1f}')
print(f'Image dimensions : {image_shape[1]} × {image_shape[0]} px')

intensity_matrix = normalize_tic(intensity_matrix)
print('TIC normalisation applied.')

In [ ]:
# ── Step 2: Peak finding and alignment ─────────────────────────────────────────
peak_mz, peak_matrix = find_and_align_peaks(
    intensity_matrix, mz_array,
    min_intensity_fraction=min_intensity_fraction,
    ppm_tolerance=ppm_tolerance
)

print(f'Peaks detected: {len(peak_mz)}')

fig = plot_mean_spectrum(mz_array, intensity_matrix, peak_mz=peak_mz)
fig.savefig(os.path.join(output_folder, 'mean_spectrum.png'), dpi=200, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Step 3: Import QuPath region annotations ────────────────────────────────────
region_masks = import_qupath_annotations(geojson_path, image_shape)

print('Annotated regions found:')
for name, mask in region_masks.items():
    print(f'  {name}: {mask.sum()} pixels')

In [ ]:
# ── Step 4: ROC analysis to identify discriminative m/z values ─────────────────
mask_a = region_masks[region_a_name]
mask_b = region_masks[region_b_name]

roc_df = compute_roc_per_mz(peak_matrix, peak_mz, mask_a, mask_b, coordinates, image_shape)
print(f'Significant m/z values (p < 0.01): {len(roc_df)}')

disc_mz_df = get_discriminative_mz(roc_df, auc_high=auc_high, auc_low=auc_low)
print(f'Discriminative m/z (AUC >= {auc_high} or <= {auc_low}): {len(disc_mz_df)}')

disc_mz_df.to_csv(os.path.join(output_folder, 'discriminative_mz.csv'), index=False)

fig = plot_roc_summary(roc_df, auc_high=auc_high, auc_low=auc_low)
fig.savefig(os.path.join(output_folder, 'roc_summary.png'), dpi=200, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Step 5: Ion images for top discriminative m/z values ───────────────────────
top_mz = disc_mz_df.head(8)['mz'].tolist()

fig = plot_ion_images(peak_matrix, peak_mz, top_mz, coordinates, image_shape)
fig.savefig(os.path.join(output_folder, 'ion_images_top_discriminative.png'),
            dpi=200, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Step 6: PCA on discriminative m/z values ───────────────────────────────────
pc_images, loadings_df, explained_variance, pca_model = run_pca(
    peak_matrix, peak_mz, disc_mz_df, coordinates, image_shape,
    n_components=n_pca_components
)

for i, ev in enumerate(explained_variance):
    print(f'PC{i+1}: {ev * 100:.2f}% variance explained')

loadings_df.to_csv(os.path.join(output_folder, 'pca_loadings.csv'), index=False)

fig = plot_pca_results(pc_images, loadings_df, explained_variance, n_components_to_show=3)
fig.savefig(os.path.join(output_folder, 'pca_results.png'), dpi=200, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# ── Step 7: Bisecting k-means segmentation ─────────────────────────────────────
segment_image, cluster_labels, cluster_areas = run_bisecting_kmeans(
    peak_matrix, peak_mz, disc_mz_df, coordinates, image_shape,
    n_clusters=n_clusters
)

print(cluster_areas.to_string(index=False))
cluster_areas.to_csv(os.path.join(output_folder, 'cluster_areas.csv'), index=False)

fig = plot_segmentation(segment_image, n_clusters)
fig.savefig(os.path.join(output_folder, 'segmentation.png'), dpi=200, bbox_inches='tight')
plt.show()
plt.close()